# Enhancement Pipeline Notebook
Apply CLAHE + Adaptive Gamma to create enhanced dataset.

In [1]:
import cv2
import numpy as np
import shutil
from pathlib import Path

In [2]:
!unzip occ_50.zip

Streaming output truncated to the last 5000 lines.
  inflating: labels/train/2015_04742.txt  
  inflating: labels/train/2015_00546.txt  
  inflating: labels/train/2015_01909.txt  
  inflating: labels/train/2015_04470.txt  
  inflating: labels/train/2015_02243.txt  
  inflating: labels/train/2015_02799.txt  
  inflating: labels/train/2015_03036.txt  
  inflating: labels/train/2015_04068.txt  
  inflating: labels/train/2015_03563.txt  
  inflating: labels/train/2015_03183.txt  
  inflating: labels/train/2015_05454.txt  
  inflating: labels/train/2015_03250.txt  
  inflating: labels/train/2015_00919.txt  
  inflating: labels/train/2015_01728.txt  
  inflating: labels/train/2015_02333.txt  
  inflating: labels/train/2015_00677.txt  
  inflating: labels/train/2015_02220.txt  
  inflating: labels/train/2015_06267.txt  
  inflating: labels/train/2015_04593.txt  
  inflating: labels/train/2015_05572.txt  
  inflating: labels/train/2015_00891.txt  
  inflating: labels/train/2015_02515.txt  
  i

In [3]:
!mkdir dataset_split
!mv images dataset_split/
!mv labels dataset_split/

In [4]:
import cv2
import numpy as np
from pathlib import Path
import shutil

# CLAHE
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)

    return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2BGR)


# Adaptive Gamma
def adaptive_gamma(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = gray.mean()

    if mean < 80:
        gamma = 0.6
    elif mean > 180:
        gamma = 1.4
    else:
        return img

    table = np.array([(i/255.0)**gamma * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, table)


# Full pipeline
def enhance_image(img):
    img = apply_clahe(img)
    img = adaptive_gamma(img)
    return img


# Apply to dataset
def enhance_dataset(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    for split in ["train", "val", "test"]:
        in_path = input_dir / "images" / split
        out_path = output_dir / "images" / split

        out_path.mkdir(parents=True, exist_ok=True)

        for img_path in in_path.glob("*"):
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            enhanced = enhance_image(img)
            cv2.imwrite(str(out_path / img_path.name), enhanced)

    print("✅ Enhancement done")


enhance_dataset("dataset_split", "dataset_enhanced")

# copy labels
shutil.copytree("dataset_split/labels", "dataset_enhanced/labels", dirs_exist_ok=True)

✅ Enhancement done


'dataset_enhanced/labels'

In [6]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00


In [7]:
yaml_content = """
path: dataset_enhanced
train: images/train
val: images/val
test: images/test

nc: 12
names: [Bicycle, Boat, Bottle, Bus, Car, Cat, Chair, Cup, Dog, Motorbike, People, Table]
"""

with open("enhanced.yaml", "w") as f:
    f.write(yaml_content)

print("✅ YAML created")

✅ YAML created


In [8]:
from ultralytics import YOLO

model_enh = YOLO("yolov8n.pt")

model_enh.train(
    data="enhanced.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    name="enhanced_model"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=enhanced.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=F

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79fbe1f80980>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,  

In [9]:
metrics_occ50 = model_enh.val(data="enhanced.yaml", split="test")

print("OCC50 TEST:")
print("mAP50:", metrics_occ50.box.map50)
print("mAP50-95:", metrics_occ50.box.map)
print("Precision:", metrics_occ50.box.mp)
print("Recall:", metrics_occ50.box.mr)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1983.3±992.7 MB/s, size: 410.7 KB)
val: Scanning /content/dataset_enhanced/labels/test... 737 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 737/737 1.9Kit/s 0.4s
val: New cache created: /content/dataset_enhanced/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 4.7it/s 10.0s
                   all        737       2437      0.767      0.761      0.812      0.549
               Bicycle         78        112      0.788      0.795      0.872      0.627
                  Boat         73        147      0.917      0.857       0.93      0.565
                Bottle         62        157      0.766       0.65      0.738      0.435
                   Bus         60         73      0.894      0.932 

In [10]:
import kagglehub

path = kagglehub.dataset_download("dakshivashishtha/exdark-occluded")

print("Path:", path)

100%|██████████| 4.85G/4.85G [01:30<00:00, 57.6MB/s]


Extracting files...
Path: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1


In [11]:
import os

print("Root:", os.listdir(path))

Root: ['occ25', 'clean', 'occ75', 'occ50']


In [12]:
base = path

def create_yaml(name, path):
    yaml = f"""
path: {path}

train: images
val: images
test: images

nc: 12
names: [Bicycle, Boat, Bottle, Bus, Car, Cat, Chair, Cup, Dog, Motorbike, People, Table]
"""
    with open(f"{name}.yaml", "w") as f:
        f.write(yaml)

create_yaml("clean", base + "/clean")
create_yaml("occ25", base + "/occ25")
create_yaml("occ75", base + "/occ75")

print("✅ YAML ready")

✅ YAML ready


In [13]:
results = {}

results["occ50_test"] = {
    "mAP50": metrics_occ50.box.map50,
    "mAP50-95": metrics_occ50.box.map,
    "Precision": metrics_occ50.box.mp,
    "Recall": metrics_occ50.box.mr
}

In [14]:
for d in ["clean", "occ25", "occ75"]:
    m = model_enh.val(data=f"{d}.yaml")

    results[d] = {
        "mAP50": m.box.map50,
        "mAP50-95": m.box.map,
        "Precision": m.box.mp,
        "Recall": m.box.mr
    }

    print(f"\n📊 {d.upper()} saved")

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 39.4±17.9 MB/s, size: 110.6 KB)
val: Scanning /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/clean/labels... 7361 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 7361/7361 534.0it/s 13.8s
val: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/clean/images/2015_05337.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [      1.075]
val: New cache created: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/clean/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 460/460 6.0it/s 1:16
                   all       7360      23702      0.513      0.324      0.346        0.2
               Bicycle        750       1118      0.777      0.303      0.421      0.253
                  Boa

In [15]:
import pandas as pd

df = pd.DataFrame(results).T

df["F1"] = 2 * (df["Precision"] * df["Recall"]) / (df["Precision"] + df["Recall"] + 1e-6)

df = df.round(4)

print("\n🔥 FINAL RESULTS")
print(df)

df.to_csv("enhanced_results.csv")


🔥 FINAL RESULTS
             mAP50  mAP50-95  Precision  Recall      F1
occ50_test  0.8120    0.5487     0.7667  0.7609  0.7638
clean       0.3464    0.1998     0.5129  0.3238  0.3970
occ25       0.1692    0.0771     0.2473  0.2549  0.2511
occ75       0.2827    0.0813     0.4021  0.3348  0.3654


In [16]:
import os

def predict_sample(folder, name, n=5):
    imgs = os.listdir(folder)[:n]
    for img in imgs:
        model_enh.predict(f"{folder}/{img}", save=True, name=name)

predict_sample("dataset_enhanced/images/test", "enh_occ50")
predict_sample(base + "/clean/images", "enh_clean")
predict_sample(base + "/occ25/images", "enh_occ25")
predict_sample(base + "/occ75/images", "enh_occ75")


image 1/1 /content/dataset_enhanced/images/test/2015_05191.jpg: 640x544 1 Dog, 53.2ms
Speed: 2.2ms preprocess, 53.2ms inference, 6.7ms postprocess per image at shape (1, 3, 640, 544)
Results saved to /content/runs/detect/enh_occ50

image 1/1 /content/dataset_enhanced/images/test/2015_07000.jpg: 448x640 1 People, 1 Table, 39.6ms
Speed: 1.2ms preprocess, 39.6ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/enh_occ50-2

image 1/1 /content/dataset_enhanced/images/test/2015_04586.jpg: 416x640 1 Cat, 1 Cup, 38.2ms
Speed: 1.0ms preprocess, 38.2ms inference, 1.3ms postprocess per image at shape (1, 3, 416, 640)
Results saved to /content/runs/detect/enh_occ50-3

image 1/1 /content/dataset_enhanced/images/test/2015_06447.jpg: 640x480 9 Peoples, 41.7ms
Speed: 1.8ms preprocess, 41.7ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/enh_occ50-4

image 1/1 /content/dataset_enhanced/images

In [17]:
import shutil

# zip model + outputs
shutil.make_archive("enhanced_outputs", 'zip', "runs/detect")

# zip CSV separately
shutil.make_archive("results_file", 'zip', ".", "enhanced_results.csv")

print("✅ All files zipped")

✅ All files zipped


In [18]:
from google.colab import files

files.download("enhanced_outputs.zip")
files.download("results_file.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import shutil

shutil.make_archive("final_output", 'zip', "runs/detect")

'/content/final_output.zip'